# 02. Data Quality Report

## Цель
Оценить качество данных IMDb перед анализом. Выявить проблемы,
которые могут повлиять на результаты.

## Чек-лист проверок
- **Полнота (Completeness)** — % NULL в ключевых полях
- **Ссылочная целостность (Referential Integrity)** — «сиротские» записи
- **Валидность (Validity)** — выбросы в числовых полях
- **Уникальность (Uniqueness)** — дубликаты в первичных ключах

## Источник данных
- Датасет: [IMDb Non-Commercial Datasets](https://datasets.imdbws.com/)
- Загружено в: PostgreSQL (schema `imdb`)

In [1]:
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text, inspect
from dotenv import load_dotenv

load_dotenv(Path("../.env"))
DB_URL = (
    f"postgresql+psycopg://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL)

## Полнота (Completeness)

### Количество строк всех таблиц

In [ ]:
tables = (inspect(engine).get_table_names(schema= 'imdb'))
count = {}
for table in tables :
    query = f"select count(*) as count from imdb.{table}"
    df = pd.read_sql(query, engine)
    count[table] = int(df['count'].iloc[0])

In [ ]:
display(count)

{'title_basics': 12812871,
 'title_principals': 101902684,
 'title_episode': 9907631,
 'title_akas': 59478714,
 'name_basics': 15681293,
 'title_ratings': 1713431,
 'title_crew': 12812871}

### Проверка на NULL

In [ ]:
tables = inspect(engine).get_table_names(schema="imdb")
for table in tables:
    columns = inspect(engine).get_columns(table, schema="imdb")
    filter = []

    for col in columns:
        name = col["name"]

        filter.append(f"""
            COUNT(*) FILTER (WHERE "{name}" IS NULL) AS "{name}"
    """
        )

    query = f"""
        SELECT
            COUNT(*) AS total_rows,
            {", ".join(filter)}
        FROM imdb.{table}
    """
    # print(query)


    with engine.connect() as conn:
        result = conn.execute(text(query)).mappings().one()

    null_summary = pd.DataFrame({
        "null_count": [
            result[col["name"]]
            for col in columns
        ]
    }, index=[
        col["name"]
        for col in columns
    ])

    null_summary["null_percent"] = (
        null_summary["null_count"]
        / result["total_rows"]
        * 100
    ).round(2)

    print(f"\n{table}")
    display(null_summary)


title_basics


,null_count,null_percent
tconst,0,0.00
titletype,0,0.00
primarytitle,0,0.00
originaltitle,0,0.00
isadult,0,0.00
startyear,1483151,11.58
endyear,12650869,98.74
runtimeminutes,8013392,62.54
genres,543489,4.24



title_principals


,null_count,null_percent
tconst,0,0.00
ordering,0,0.00
nconst,0,0.00
category,0,0.00
job,82258930,80.72
characters,52223487,51.25



title_episode


...

,null_count,null_percent
nconst,0,0.00
primaryname,88,0.00
birthyear,14999010,95.65
deathyear,15417772,98.32
primaryprofession,3180110,20.28
knownfortitles,1895401,12.09



title_ratings


,null_count,null_percent
tconst,0,0.0
averagerating,0,0.0
num_votes,0,0.0



title_crew


,null_count,null_percent
tconst,0,0.00
directors,5696680,44.46
writers,6334214,49.44


### Просмотр минимальных и максимальных значений 

In [ ]:
tables = inspect(engine).get_table_names(schema="imdb")
numeric = ('INTEGER','SMALLINT','NUMERIC',"REAL","DOUBLE")
for table in tables:
    columns = inspect(engine).get_columns(table, schema="imdb")
    print('\n' + table + '\n')
    num_col = [
    col['name']
    for col in columns
        if str(col['type']).upper().startswith(numeric)]
    #print(num_col)
    if num_col:
        query = f"""
    select {",".join(
        f'MIN("{col}") as "{col}_min", '
        f'MAX("{col}") as "{col}_MAX"'
        for col in num_col
    )}
    FROM imdb."{table}"
"""
        if query:
            df_min_max = pd.read_sql(query,engine)
            display(df_min_max)

    



title_basics



,startyear_min,startyear_MAX,endyear_min,endyear_MAX,runtimeminutes_min,runtimeminutes_MAX
0,1874,2115,1928,2035,0,3692080



title_principals



,ordering_min,ordering_MAX
0,1,69



title_episode



...


name_basics



,birthyear_min,birthyear_MAX,deathyear_min,deathyear_MAX
0,4,2026,17,2026



title_ratings



,averagerating_min,averagerating_MAX,num_votes_min,num_votes_MAX
0,1.0,10.0,5,3242062



title_crew



## Ссылочная целостность (Referential Integrity)

In [ ]:
# Проверка ссылочной целостности по всем FK
# Формат: (описание, дочерняя таблица, столбец дочерней, 
#           родительская таблица, столбец родительской)

checks = [
    ("title_ratings.tconst → title_basics.tconst", 
     "title_ratings", "tconst", "title_basics", "tconst"),

    ("title_crew.tconst → title_basics.tconst",
     "title_crew", "tconst", "title_basics", "tconst"),

    ("title_principals.tconst → title_basics.tconst",
     "title_principals", "tconst", "title_basics", "tconst"),

    ("title_principals.nconst → name_basics.nconst",
     "title_principals", "nconst", "name_basics", "nconst"),

    ("title_episode.tconst → title_basics.tconst",
     "title_episode", "tconst", "title_basics", "tconst"),

    ("title_episode.parenttconst → title_basics.tconst",
     "title_episode", "parenttconst", "title_basics", "tconst"),
     
    ("title_akas.titleid → title_basics.tconst",
     "title_akas", "titleid", "title_basics", "tconst"),
]

results = []

for desc, child, child_col, parent, parent_col in checks:
    query = f"""
    SELECT 
        COUNT(*) FILTER (WHERE p.{parent_col} IS NULL) AS orphans,
        COUNT(*) AS total,
        ROUND(COUNT(*) FILTER (WHERE p.{parent_col} IS NULL) * 100.0 / COUNT(*), 4) AS pct
    FROM imdb.{child} c
    LEFT JOIN imdb.{parent} p ON c.{child_col} = p.{parent_col}
    """
    
    row = pd.read_sql(query, engine).iloc[0]
    results.append({
        "Связь": desc,
        "Сироты": int(row["orphans"]),
        "Всего": int(row["total"]),
        "% сирот": row["pct"],
    })

orphan_report = pd.DataFrame(results)
orphan_report

,Связь,Сироты,Всего,% сирот
0,title_ratings.tconst → title_basics.tconst,0,1713431,0.0000
1,title_crew.tconst → title_basics.tconst,0,12812871,0.0000
2,title_principals.tconst → title_basics.tconst,0,101902684,0.0000
3,title_principals.nconst → name_basics.nconst,7116,101902684,0.0070
4,title_episode.tconst → title_basics.tconst,0,9907631,0.0000
5,title_episode.parenttconst → title_basics.tconst,10,9907631,0.0001
6,title_akas.titleid → title_basics.tconst,0,59478714,0.0000


## Валидность (Validity)

### Аномально длинные произведения 


In [ ]:
query = """
SELECT primarytitle, titletype, startyear, runtimeminutes
FROM imdb.title_basics
WHERE runtimeminutes > 1440
ORDER BY runtimeminutes DESC
LIMIT 10;
"""
display(pd.read_sql(query, engine))

# Сколько всего таких
query = """
SELECT 
    COUNT(*) AS suspicious_count,
    ROUND(AVG(runtimeminutes), 0) AS avg_runtime
FROM imdb.title_basics
WHERE runtimeminutes > 1440;
"""
display(pd.read_sql(query, engine))

,primarytitle,titletype,startyear,runtimeminutes
0,The Angry Grandpa Show,tvSeries,2007,3692080
1,Love Link TV the Mixer,tvSeries,2024,1000000
2,Shree Haricharitramrut Sagar,video,2017,146445
3,100,video,2019,59460
4,Logistics,movie,2012,51420
5,Ambiancé,movie,2020,43200
6,Unsual What if (Old Videos),tvMiniSeries,2023,37440
7,The Longest Video on YouTube: 596.5 Hours,video,2011,35791
8,Le Sacre du printemps,tvMovie,2026,30240
9,Carnets Filmés (Liste Complète),movie,2019,28643


,suspicious_count,avg_runtime
0,204,28128.0


#### Группировка по типам произведений

In [ ]:
query = """
SELECT 
    titletype,
    COUNT(*) AS cnt,
    MIN(runtimeminutes) AS min_rt,
    MAX(runtimeminutes) AS max_rt,
    ROUND(AVG(runtimeminutes), 0) AS avg_rt
FROM imdb.title_basics
WHERE runtimeminutes > 1440
GROUP BY titletype
ORDER BY cnt DESC;
"""
display(pd.read_sql(query, engine))

,titletype,cnt,min_rt,max_rt,avg_rt
0,video,63,1445,146445,7000.0
1,tvSeries,55,1442,3692080,87931.0
2,movie,29,1452,51420,7397.0
3,tvSpecial,21,1441,27225,3980.0
4,tvEpisode,21,1500,6000,2432.0
5,tvMovie,7,1500,30240,6600.0
6,tvMiniSeries,6,1552,37440,10370.0
7,videoGame,2,1495,1800,1648.0


### Пока не вышедшие произведения

In [ ]:
query = """
SELECT 
    COUNT(*) AS future_titles,
    MIN(startyear) AS min_year,
    MAX(startyear) AS max_year
FROM imdb.title_basics
WHERE startyear > EXTRACT(YEAR FROM CURRENT_DATE);
"""
display(pd.read_sql(query, engine))

,future_titles,min_year,max_year
0,3785,2027,2115


## Уникальность (Uniqueness)

In [ ]:
pk_checks = {
    "title_basics": "tconst",
    "title_ratings": "tconst",
    "name_basics": "nconst",
    "title_crew": "tconst",
    "title_episode": "tconst",
}

for table, pk in pk_checks.items():
    query = f"""
    SELECT COUNT(*) AS duplicates
    FROM (
        SELECT {pk}, COUNT(*) AS cnt
        FROM imdb.{table}
        GROUP BY {pk}
        HAVING COUNT(*) > 1
    ) sub;
    """
    dup_count = pd.read_sql(query, engine).iloc[0]["duplicates"]
    
    status = "OK" if dup_count == 0 else f"{dup_count} дубликатов"
    print(f"{table} ({pk}): {status}")

title_basics (tconst): OK
title_ratings (tconst): OK
name_basics (nconst): OK
title_crew (tconst): OK
title_episode (tconst): OK


## Выводы 

### Общая оценка

Датасет IMDb содержит **около 240 млн записей** в 7 таблицах. 
Общее качество данных — **высокое**, ключевые проблемы локализованы 
и не повлияют на дальнейший анализ.

### Найденные проблемы

**1. Полнота (Completeness)**

- `endYear` заполнен только у **1.26%** записей — это ожидаемо, так как 
  поле применимо только к сериалам, у которых есть год окончания.
- `runtimeMinutes` пуст у **62.54%** — характерно для эпизодов сериалов 
  и короткометражек, для которых длительность часто не указывается.
- `job` в `title_principals` пуст у **80.72%** — заполняется только 
  для узких специальностей (например, "camera operator"), 
  для актёров поле не применяется.

**Вывод:** высокая доля NULL в этих полях — не дефект данных, 
а особенность предметной области.

**2. Ссылочная целостность (Referential Integrity)**

Проверено 7 связей между таблицами. Обнаружено:

| Связь | Сироты | % |
|-------|--------|---|
| `title_principals.nconst → name_basics.nconst` | 7,116 | 0.007% |
| `title_episode.parenttconst → title_basics.tconst` | 10 | 0.0001% |
| Остальные 5 связей | 0 | 0% |

**Всего 7 126 сиротских записей из примерно 285 млн (0.0025%).**

**Причина:** данные в разных файлах IMDb обновляются в разное время.

**3. Валидность (Validity)**

- **Аномально длинные произведения:** обнаружено **204 записи** 
  с `runtimeMinutes > 1440` (> 24 часов). Максимум — 3 692 080 минут 
  (~7 лет) у шоу "The Angry Grandpa Show".
 
 **Важно:** это не ошибки данных, а реальные значения:
  - `tvSeries` — `runtimeMinutes` указывает **суммарную длительность всех эпизодов**, 
    а не одного эпизода. Поэтому "The Angry Grandpa Show" (2007) 
    с 3.6 млн минут — это сумма всех выпусков.
  - `movie` — существуют **экспериментальные фильмы** с реальной длительностью 
    в десятки часов (например, "Logistics" — 35 дней, "Ambiancé" — 30 дней).
  - `video` — есть **сверхдлинные видео на YouTube** (например, 596-часовое видео).

  **Для анализа:** эти записи **исключаем** из анализа длительности, 
  так как они искажают средние показатели (204 записи из 12.8 млн = 0.0016%, 
  но они сильно тянут среднее вверх).

**4. Уникальность (Uniqueness)**

Все первичные ключи уникальны, дубликатов нет.